# Simplifying Regular Expressions with Flattened Sums and Products

The notebook `Rewrite.ipynb` simplifies regular expressions by rewriting them with rules like
$R + \emptyset \rightarrow R$.  These rules work on the syntax tree of a regular expression, where a sum
$r_1 + r_2 + r_3$ is either the tree $(r_1 + r_2) + r_3$ or the tree $r_1 + (r_2 + r_3)$.  A rule matches only
one of these trees.  Furthermore, two summands of a sum are only compared if they happen to be neighbours in
the tree.  Therefore, the sum $\texttt{b} + \texttt{a} + \texttt{b}^*$ is not simplified to $\texttt{a} + \texttt{b}^*$,
although $L(\texttt{b}) \subseteq L(\texttt{b}^*)$.

In this notebook we represent a sum as the *set* of its summands and a product as the *tuple* of its factors.
Then the laws of associativity, commutativity, and idempotence of $+$, i.e.
$$ (r_1 + r_2) + r_3 \doteq r_1 + (r_2 + r_3), \quad r_1 + r_2 \doteq r_2 + r_1, \quad\mbox{and}\quad r + r \doteq r, $$
and the law of associativity of $\cdot$ are built into the representation.  The other laws from Chapter 2 of
the lecture notes are applied by *smart constructors*:  These are functions that build a sum, a product, or a
Kleene closure and simplify it at the same time.  Since the smart constructors are applied bottom-up, every
subexpression is already simplified when it becomes part of a larger expression.  Therefore, a single pass over
the regular expression suffices.

In [ ]:
from functools import cache
from typing import Literal

The type `RegExp` describes regular expressions as they are used in the other notebooks:  `0` stands for
$\emptyset$, the string `'𝜀'` stands for $\varepsilon$, a string of length $1$ is a character, and
nested tuples represent the operators `'*'`, `'⋅'`, and `'+'`.

In [ ]:
type BinaryOp = Literal['⋅', '+']
type UnaryOp  = Literal['*']
type RegExp   = int | str | tuple[RegExp, UnaryOp] | tuple[RegExp, BinaryOp, RegExp]

The type `Flat` describes the *flattened* representation of regular expressions:
- A character is represented by itself.
- The Kleene closure $r^*$ is represented as the pair `('*', r)`.
- The sum $r_1 + \cdots + r_n$ is represented as the pair `('+', frozenset({r1, ..., rn}))`.  None of the
  summands $r_i$ is itself a sum.
- The product $r_1 \cdot \ldots \cdot r_n$ is represented as the pair `('⋅', (r1, ..., rn))`.  None of the
  factors $r_i$ is itself a product.

The empty regular expression $\emptyset$ is the empty sum and $\varepsilon$ is the empty product, since $\emptyset$
is the neutral element of $+$ and $\varepsilon$ is the neutral element of $\cdot$.  Apart from these two, every
sum has at least two summands and every product has at least two factors.

In [ ]:
type Flat = (str | tuple[Literal['*'], Flat] | tuple[Literal['+'], frozenset[Flat]]
                 | tuple[Literal['⋅'], tuple[Flat, ...]])

Empty:   Flat = ('+', frozenset())
Epsilon: Flat = ('⋅', ())

## Pretty Printing and Size

The functions `precedence`, `arg_str`, and `to_str` are adapted from the notebook `Chapter-02/Regular-Expressions.ipynb`.
The function `to_str(r)` turns the regular expression `r` into a string that only contains the parentheses
that are needed.

**Function `precedence(r)`**
- *Input:* `r` is a regular expression, represented as a nested tuple.
- *Output:* The precedence of the operator at the root of `r`.  Atoms have the precedence $4$.

In [ ]:
def precedence(r: RegExp) -> int:
    if not isinstance(r, tuple):
        return 4
    return { '*': 3, '⋅': 2, '+': 1 }[r[1]]

**Function `arg_str(r, p)`**
- *Input:* `r` is a regular expression and `p` is a precedence.
- *Output:* The string representing `r`, enclosed in parentheses if the precedence of `r` is lower than `p`.

In [ ]:
def arg_str(r: RegExp, p: int) -> str:
    if precedence(r) < p:
        return '(' + to_str(r) + ')'
    return to_str(r)

**Function `to_str(r)`**
- *Input:* `r` is a regular expression, represented as a nested tuple.
- *Output:* The string representing `r`, using as few parentheses as possible.

In [ ]:
def to_str(r: RegExp) -> str:
    match r:
        case 0:
            return '∅'
        case str():
            return r
        case (r1, '*'):
            return arg_str(r1, 3) + '*'
        case (r1, op, r2):
            p   = precedence(r)
            sep = ' + ' if op == '+' else '⋅'
            return arg_str(r1, p) + sep + arg_str(r2, p + 1)
        case _:
            raise ValueError(f'{r} is not a proper regular expression.')

To measure the size of a regular expression we count the nodes of its syntax tree:  Every character, every
occurrence of $\varepsilon$ and $\emptyset$, and every operator counts as one node.

**Function `size(r)`**
- *Input:* `r` is a regular expression, represented as a nested tuple.
- *Output:* The number of nodes of the syntax tree of `r`.

In [ ]:
def size(r: RegExp) -> int:
    match r:
        case (r1, '*'):
            return 1 + size(r1)
        case (r1, _, r2):
            return 1 + size(r1) + size(r2)
        case _:
            return 1

The function `flat_size(r)` computes the size of a flattened regular expression `r`.  This is the size of the
syntax tree that `r` represents:  A sum of $n$ summands needs $n - 1$ operators `+` and a product of $n$ factors
needs $n - 1$ operators `⋅`.

**Function `flat_size(r)`**
- *Input:* `r` is a flattened regular expression.
- *Output:* The number of nodes of the syntax tree that `r` represents.

In [ ]:
@cache
def flat_size(r: Flat) -> int:
    match r:
        case ('*', s):
            return 1 + flat_size(s)
        case ('+', S) if len(S) > 0:
            return sum(flat_size(s) for s in S) + len(S) - 1
        case ('⋅', F) if len(F) > 0:
            return sum(flat_size(f) for f in F) + len(F) - 1
        case _:
            return 1

## Auxiliary Functions

The function `summands(r)` returns the set of summands of `r`.  If `r` is not a sum, `r` is its only summand.
Similarly, the function `factors(r)` returns the tuple of factors of `r`.

**Function `summands(r)`**
- *Input:* `r` is a flattened regular expression.
- *Output:* The set of summands of `r`.

In [ ]:
def summands(r: Flat) -> frozenset[Flat]:
    if isinstance(r, tuple) and r[0] == '+':
        return r[1]
    return frozenset({ r })

**Function `factors(r)`**
- *Input:* `r` is a flattened regular expression.
- *Output:* The tuple of factors of `r`.

In [ ]:
def factors(r: Flat) -> tuple[Flat, ...]:
    if isinstance(r, tuple) and r[0] == '⋅':
        return r[1]
    return (r,)

A regular expression $r$ is *nullable* iff $\varepsilon \in L(r)$.  A Kleene closure is always nullable, a sum
is nullable iff one of its summands is nullable, and a product is nullable iff all of its factors are nullable.
In particular, $\varepsilon$ is nullable as the empty product, while $\emptyset$ is not nullable as the empty sum.

**Function `nullable(r)`**
- *Input:* `r` is a flattened regular expression.
- *Output:* `True` if $\varepsilon \in L(r)$, `False` otherwise.

In [ ]:
@cache
def nullable(r: Flat) -> bool:
    match r:
        case ('*', _):
            return True
        case ('+', S):
            return any(nullable(s) for s in S)
        case ('⋅', F):
            return all(nullable(f) for f in F)
        case _:
            return False

The function `included(r1, r2)` checks whether $L(r_1) \subseteq L(r_2)$.  It only inspects the syntax of
$r_1$ and $r_2$.  If it returns `True`, then $L(r_1) \subseteq L(r_2)$ holds.  If it returns `False`, the inclusion
might still hold.  The function uses the following facts:
1. $L(r) \subseteq L(r)$ and $L(\emptyset) \subseteq L(r)$.
2. $L(\varepsilon) \subseteq L(r)$ iff $r$ is nullable.
3. $L(s_1 + \cdots + s_n) \subseteq L(r)$ iff $L(s_i) \subseteq L(r)$ for all $i$.
4. If $L(r) \subseteq L(s_i)$ for some $i$, then $L(r) \subseteq L(s_1 + \cdots + s_n)$.
5. If $L(r) \subseteq L(s)$, then $L(r) \subseteq L(s^*)$.  If $L(t) \subseteq L(s^*)$, then also
   $L(t^*) \subseteq L(s^*)$.  If $L(f_i) \subseteq L(s^*)$ for all $i$, then
   $L(f_1 \cdot \ldots \cdot f_n) \subseteq L(s^*)$, since $L(s^*)$ is closed under concatenation.

**Function `included(r1, r2)`**
- *Input:* `r1` and `r2` are flattened regular expressions.
- *Output:* `True` if the syntax of `r1` and `r2` shows that $L(r_1) \subseteq L(r_2)$, `False` otherwise.

In [ ]:
@cache
def included(r1: Flat, r2: Flat) -> bool:
    if r1 == r2 or r1 == Empty:
        return True
    if r1 == Epsilon:
        return nullable(r2)
    match r1:
        case ('+', S1):
            return all(included(s, r2) for s in S1)
        case _:
            pass
    match r2:
        case ('+', S2):
            return any(included(r1, s) for s in S2)
        case ('*', s2):
            if included(r1, s2):
                return True
            match r1:
                case ('*', s1):
                    return included(s1, r2)
                case ('⋅', F1):
                    return all(included(f, r2) for f in F1)
                case _:
                    return False
        case _:
            return False

Since sets are unordered, the smart constructors sort summands with the function `sort_key` whenever the
result depends on the order in which they are processed.  This makes the results independent of the order
in which Python stores the elements of a set.  The key sorts by size first and then by the string
representation.

**Function `sort_key(r)`**
- *Input:* `r` is a flattened regular expression.
- *Output:* The pair consisting of the size of `r` and the string representing `r`.

In [ ]:
@cache
def sort_key(r: Flat) -> tuple[int, str]:
    return flat_size(r), to_str(unflatten(r))

## Smart Constructors for Sums

If $L(x) \subseteq L(y)$ holds for two summands $x$ and $y$ of a sum, then $x + y \doteq y$ and the summand $x$
can be dropped.  For example, $\texttt{b} + \texttt{b}^* \doteq \texttt{b}^*$ and $\varepsilon + r^* \doteq r^*$.
If both $L(x) \subseteq L(y)$ and $L(y) \subseteq L(x)$, only the summand with the smaller key is kept.

**Function `absorb(S)`**
- *Input:* `S` is a set of summands.
- *Output:* The set of those summands of `S` that are not contained in another summand.

In [ ]:
def absorb(S: frozenset[Flat]) -> frozenset[Flat]:
    return frozenset(x for x in S
                       if not any(y != x and included(x, y) and
                                  (not included(y, x) or sort_key(y) < sort_key(x))
                                  for y in S)
                    )

The law $\varepsilon + r^* \cdot r \doteq r^*$ from Chapter 2 and its mirror image
$\varepsilon + r \cdot r^* \doteq r^*$ replace the summands $\varepsilon$ and $r \cdot r^*$ by $r^*$.

**Function `plus_star(S)`**
- *Input:* `S` is a set of summands.
- *Output:* If `S` contains $\varepsilon$ and a product of the form $r \cdot r^*$ or $r^* \cdot r$, these two
  summands are replaced by $r^*$.  Otherwise, `S` is returned unchanged.

In [ ]:
def plus_star(S: frozenset[Flat]) -> frozenset[Flat]:
    if Epsilon not in S:
        return S
    for x in sorted(S, key=sort_key):
        match factors(x):
            case (f, ('*', g)) | (('*', g), f) if f == g:
                star: Flat = ('*', f)
                return S - { Epsilon, x } | { star }
            case _:
                pass
    return S

The laws of distributivity
$$ s \cdot t_1 + s \cdot t_2 \doteq s \cdot (t_1 + t_2) \quad\mbox{and}\quad
   t_1 \cdot s + t_2 \cdot s \doteq (t_1 + t_2) \cdot s $$
are used to factor out common prefixes and suffixes.  A summand $s$ that is not a product counts as the product
$s \cdot \varepsilon$, e.g. $\texttt{a} + \texttt{a} \cdot \texttt{b}^*$ is turned into
$\texttt{a} \cdot (\varepsilon + \texttt{b}^*) \doteq \texttt{a} \cdot \texttt{b}^*$.  Factoring out does not always make
a regular expression smaller.  Therefore, the function `factor_out` tries every first factor and every last factor
that is shared by several summands and returns the smallest of the resulting sums.

**Function `factor_out(S)`**
- *Input:* `S` is a set of at least two summands.
- *Output:* A flattened regular expression that is equivalent to the sum of `S` and is as small as possible
  after factoring out one common first or last factor.

In [ ]:
def factor_out(S: frozenset[Flat]) -> Flat:
    best: Flat = ('+', S)
    for pos in (0, -1):
        Groups: dict[Flat, list[Flat]] = {}
        for x in sorted(S, key=sort_key):
            if x != Epsilon:
                Groups.setdefault(factors(x)[pos], []).append(x)
        for f in sorted(Groups, key=sort_key):
            xs = Groups[f]
            if len(xs) < 2:
                continue
            if pos == 0:
                rests = frozenset(mk_prod(factors(x)[1:]) for x in xs)
                new   = mk_prod((f, mk_sum(rests)))
            else:
                rests = frozenset(mk_prod(factors(x)[:-1]) for x in xs)
                new   = mk_prod((mk_sum(rests), f))
            candidate = mk_sum(S - set(xs) | { new })
            if flat_size(candidate) < flat_size(best):
                best = candidate
    return best

The smart constructor `mk_sum(S)` builds the sum of the regular expressions in the set `S`:
1. Summands that are sums themselves are replaced by their summands.  As $\emptyset$ is the empty sum, it vanishes.
2. Summands that are contained in other summands are dropped.
3. The law $\varepsilon + r \cdot r^* \doteq r^*$ is applied.
4. A sum with only one summand is this summand and a sum without summands is $\emptyset$.
5. Common prefixes and suffixes are factored out.

**Function `mk_sum(S)`**
- *Input:* `S` is a set of flattened regular expressions.
- *Output:* A simplified flattened regular expression that is equivalent to the sum of the elements of `S`.

In [ ]:
@cache
def mk_sum(S: frozenset[Flat]) -> Flat:
    S = frozenset(s for x in S for s in summands(x))
    S = plus_star(absorb(S))
    if len(S) == 0:
        return Empty
    if len(S) == 1:
        r, = S
        return r
    return factor_out(S)

## Smart Constructors for Products and Kleene Closures

The smart constructor `mk_prod(F)` builds the product of the regular expressions in the tuple `F`:
1. If one of the factors is $\emptyset$, the product is $\emptyset$.
2. Factors that are products themselves are replaced by their factors.  As $\varepsilon$ is the empty product,
   it vanishes.
3. If $y$ is a nullable factor next to a factor $x^*$ and $L(y) \subseteq L(x^*)$, then $y$ is dropped, since
   $$ L(x^*) \subseteq L(x^* \cdot y) \subseteq L(x^* \cdot x^*) = L(x^*). $$
   For example, $r^* \cdot (r + \varepsilon) \doteq r^*$ and $r^* \cdot r^* \doteq r^*$.
4. A product with only one factor is this factor.

**Function `mk_prod(F)`**
- *Input:* `F` is a tuple of flattened regular expressions.
- *Output:* A simplified flattened regular expression that is equivalent to the product of the elements of `F`.

In [ ]:
@cache
def mk_prod(F: tuple[Flat, ...]) -> Flat:
    if Empty in F:
        return Empty
    Factors = [f for x in F for f in factors(x)]
    i = 0
    while i < len(Factors) - 1:
        x, y = Factors[i], Factors[i+1]
        if isinstance(x, tuple) and x[0] == '*' and nullable(y) and included(y, x):
            del Factors[i+1]
        elif isinstance(y, tuple) and y[0] == '*' and nullable(x) and included(x, y):
            del Factors[i]
            i = max(i - 1, 0)
        else:
            i += 1
    if len(Factors) == 1:
        return Factors[0]
    return ('⋅', tuple(Factors))

Inside a Kleene closure, many regular expressions can be replaced by simpler ones.  The function `strip(r)`
computes a regular expression $s$ such that $s^* \doteq r^*$ using the following laws:
1. $(r^*)^* \doteq r^*$.
2. $(\varepsilon + r)^* \doteq r^*$ and, more generally, $(r_1 + \cdots + r_n)^* \doteq (s_1 + \cdots + s_n)^*$ if
   $s_i^* \doteq r_i^*$ for all $i$, since $(r_1 + r_2)^* \doteq (r_1^* + r_2^*)^*$.  For example,
   $(r^* + s)^* \doteq (r + s)^*$.
3. If all factors $r_1$, $\cdots$, $r_n$ are nullable, then $(r_1 \cdot \ldots \cdot r_n)^* \doteq (r_1 + \cdots + r_n)^*$,
   since every $r_i$ is contained in the product.  For example, $(\texttt{a}^* \cdot \texttt{b}^*)^* \doteq (\texttt{a} + \texttt{b})^*$.

As $\varepsilon$ is the empty product, the third law turns $\varepsilon$ into the empty sum $\emptyset$.

**Function `strip(r)`**
- *Input:* `r` is a flattened regular expression.
- *Output:* A flattened regular expression $s$ such that $s^* \doteq r^*$.

In [ ]:
def strip(r: Flat) -> Flat:
    match r:
        case ('*', s):
            return strip(s)
        case ('+', S):
            return mk_sum(frozenset(strip(s) for s in S))
        case ('⋅', F) if all(nullable(f) for f in F):
            return mk_sum(frozenset(strip(f) for f in F))
        case _:
            return r

The smart constructor `mk_star(r)` builds the Kleene closure of `r`.  Since $\emptyset^* \doteq \varepsilon$ and
$\varepsilon^* \doteq \varepsilon$, the result is $\varepsilon$ if `strip(r)` is $\emptyset$ or $\varepsilon$.

**Function `mk_star(r)`**
- *Input:* `r` is a flattened regular expression.
- *Output:* A simplified flattened regular expression that is equivalent to $r^*$.

In [ ]:
@cache
def mk_star(r: Flat) -> Flat:
    s = strip(r)
    if s == Empty or s == Epsilon:
        return Epsilon
    return ('*', s)

## Simplification

The function `flatten(r)` converts a regular expression `r` into the flattened representation.  It traverses
`r` bottom-up and builds the result with the smart constructors.  Hence the result is already simplified.

**Function `flatten(r)`**
- *Input:* `r` is a regular expression, represented as a nested tuple.
- *Output:* A simplified flattened regular expression that is equivalent to `r`.

In [ ]:
def flatten(r: RegExp) -> Flat:
    match r:
        case 0:
            return Empty
        case '𝜀':
            return Epsilon
        case str():
            return r
        case (r1, '*'):
            return mk_star(flatten(r1))
        case (r1, '+', r2):
            return mk_sum(frozenset({ flatten(r1), flatten(r2) }))
        case (r1, '⋅', r2):
            return mk_prod((flatten(r1), flatten(r2)))
        case _:
            raise ValueError(f'{r} is not a proper regular expression.')

The function `unflatten(r)` converts a flattened regular expression back into a nested tuple.  Sums and
products become left-nested trees, as they are produced by the function `parse`.  The summands of a sum are
sorted by their keys and $\varepsilon$ is put at the end, so that the result does not depend on the order of the
elements of a set.

**Function `unflatten(r)`**
- *Input:* `r` is a flattened regular expression.
- *Output:* The regular expression represented by `r` as a nested tuple.

In [ ]:
def unflatten(r: Flat) -> RegExp:
    match r:
        case ('*', s):
            return (unflatten(s), '*')
        case ('+', S):
            if len(S) == 0:
                return 0
            Rs = [unflatten(s) for s in sorted(S - { Epsilon }, key=sort_key)]
            if Epsilon in S:
                Rs.append('𝜀')
            result = Rs[0]
            for s in Rs[1:]:
                result = (result, '+', s)
            return result
        case ('⋅', F):
            if len(F) == 0:
                return '𝜀'
            result = unflatten(F[0])
            for f in F[1:]:
                result = (result, '⋅', unflatten(f))
            return result
        case _:
            return r

**Function `simplify(r)`**
- *Input:* `r` is a regular expression, represented as a nested tuple.
- *Output:* A simplified regular expression that is equivalent to `r`.

In [ ]:
def simplify(r: RegExp) -> RegExp:
    return unflatten(flatten(r))